# ARMADO DEL DATASET POST API

# DATOS

“El dataset base proviene de Make Music Equal (licencia CC BY-SA 4.0).
Este fue enriquecido con datos obtenidos mediante la API de Chartmetric, los cuales poseen restricciones de uso.
Por este motivo, las variables derivadas de dicha fuente fueron transformadas y agregadas de forma tal que no permiten reconstruir los valores originales.”

PASO 1: Base de artistas para obtener IDs antes de conectar a la API
Fuente: Make Music Equal (CC BY-SA 4.0)
La base de datos de artistas se descargó del dataset de acceso público MakeMusicEqual versión 3/3/2026: 936420, 10.
https://chartmetric-public.s3.us-west-2.amazonaws.com/make-music-equal/mme_artist_info.csv 
Se eliminaron registros duplicados [nombre + url de artista]: 172
Resultado: 936248 filas/artistas. 
Legal: This work is licensed under a Creative Commons Attribution-ShareAlike 4.0 International License.


PASO 2: Construcción de las cohortes de datos a extraer
Base de datos MakeMusicEqual ordenada por Chartmetric_rank (rank 1 a 936248)
-cohorte 1: 1400 registros 
-cohorte 2: 5600 registros 

Notas:
Hay discontinuidades en el atributo Chartmetric_rank (faltan números en la secuencia) porque la base makemusicequal está actualizada al 3/3/26 aunque fueron descargados el 23/3/2026. Misma fecha de acceso a la API. Como la variable chartmetrick_rank es dinámica, esa diferencia de 20 días hace que un artista pueda tener rank = 5 el 3/3/2026 y rank = 9 el 23/3/26. Sin embargo, eso no invalida la selección del conjunto de artistas sobre la cual se construye la extracción de datos porque el objetivo fue definir un conjunto de artistas ordenados por popularidad (medida por una fuenta de datos confiable). Además la variable rank no será utilizada en el análisis. Y por último, se comprueba que el rank puede variar sutilmente pero el conjunto de artistas definidos es predominantemente coincidente. 

PASO 3: API. Extracción de datos
Límite 14 días, 100 request x día. (inicio 23/3/26)
Se apunta a dos endpoints: artist metadata y artist live events
La extracción de datos se fragmenta por cohortes y se va guardando en archivos parciales

POSTERIORMENTE se recuperó más info del jason:
Band
y
num_sp_editorial_playlists",
    "num_sp_playlists",
    "sp_editorial_playlist_total_reach",
    "num_am_editorial_playlists",
    "num_am_playlists",
    "num_de_editorial_playlists",
    "num_de_playlists",
    "de_playlist_total_reach",
    "de_editorial_playlist_total_reach",
    "num_az_editorial_playlists",
    "num_az_playlists",
    "num_yt_editorial_playlists",
    "num_yt_playlists",
    "yt_playlist_total_reach",
    "tiktok_top_video_views",
    "tiktok_top_video_comments",
    "tiktok_track_posts"

desdargas cohorte 1: 23 a 25 de marzo (ajustando fecha de live events)
el listado de artistas lo saco de mmequal ordenado por rank eso es del (6 de marzo)


In [1]:
import pandas as pd
import numpy as np

# live events agregado

In [2]:
ruta_eventlevel = r"chartmetric_liveevents_2024_2025_cohorte2\liveevents_2024_2025_eventlevel_cohorte2.csv"

In [3]:
df_events = pd.read_csv(ruta_eventlevel)

In [4]:
print(df_events.shape)
print(df_events.columns.tolist())
df_events.head(3)

(83572, 24)
['downloaded_at_utc', 'chartmetric_id', 'artist_name', 'event_id', 'event_name', 'type', 'event_year', 'start_date', 'end_date', 'venue_id', 'venue_name', 'venue_capacity', 'city_id', 'city_name', 'code2', 'price', 'low_price', 'high_price', 'currency', 'is_headliner', 'jambase_event_url', 'songkick_event_url', 'seatgeek_event_url', 'ticketmaster_event_url']


In [5]:
# fechas
df_events["start_date"] = pd.to_datetime(df_events["start_date"], errors="coerce")
df_events["end_date"] = pd.to_datetime(df_events["end_date"], errors="coerce")

# por seguridad, si event_year vino mal tipado
df_events["event_year"] = pd.to_numeric(df_events["event_year"], errors="coerce")

# columnas numéricas que vamos a usar
cols_numericas = ["low_price", "high_price", "venue_capacity"]

for col in cols_numericas:
    df_events[col] = pd.to_numeric(df_events[col], errors="coerce")

In [6]:
print(df_events["event_year"].value_counts(dropna=False).sort_index())

event_year
2024    30459
2025    53112
2026        1
Name: count, dtype: int64


In [7]:
def moneda_principal(serie):
    serie_limpia = serie.dropna()
    if serie_limpia.empty:
        return np.nan
    return serie_limpia.mode().iloc[0]

In [8]:
def resumir_live_por_anio(df, anio):
    df_anio = df[df["event_year"] == anio].copy()
    
    resumen = (
        df_anio
        .groupby(["chartmetric_id", "artist_name"], as_index=False)
        .agg(
            n_shows=("event_id", "nunique"),
            avg_low_price=("low_price", "mean"),
            avg_high_price=("high_price", "mean"),
            avg_venue_capacity=("venue_capacity", "mean"),
            total_capacity=("venue_capacity", "sum"),
            n_shows_with_capacity=("venue_capacity", lambda x: x.notna().sum()),
            n_cities=("city_name", lambda x: x.dropna().nunique()),
            n_countries=("code2", lambda x: x.dropna().nunique()),
            n_currencies=("currency", lambda x: x.dropna().nunique()),
            main_currency=("currency", moneda_principal)
        )
    )
    
    resumen = resumen.rename(columns={
        "n_shows": f"n_shows_{anio}",
        "avg_low_price": f"avg_low_price_{anio}",
        "avg_high_price": f"avg_high_price_{anio}",
        "avg_venue_capacity": f"avg_venue_capacity_{anio}",
        "total_capacity": f"total_capacity_{anio}",
        "n_shows_with_capacity": f"n_shows_with_capacity_{anio}",
        "n_cities": f"n_cities_{anio}",
        "n_countries": f"n_countries_{anio}",
        "n_currencies": f"n_currencies_{anio}",
        "main_currency": f"main_currency_{anio}"
    })
    
    # si no hay ningún dato de capacidad, dejar total_capacity como NaN
    col_total = f"total_capacity_{anio}"
    col_ncap = f"n_shows_with_capacity_{anio}"
    resumen.loc[resumen[col_ncap] == 0, col_total] = np.nan
    
    return resumen

In [9]:
df_live_2024 = resumir_live_por_anio(df_events, 2024)
df_live_2025 = resumir_live_por_anio(df_events, 2025)

print("df_live_2024:", df_live_2024.shape)
print("df_live_2025:", df_live_2025.shape)

df_live_2024: (2396, 12)
df_live_2025: (2738, 12)


In [10]:
df_live_agg = df_live_2024.merge(
    df_live_2025,
    on=["chartmetric_id", "artist_name"],
    how="outer"
)

In [11]:
# imputo 0 para poder sumar las columnas bianuales. si hay na la suma queda na.and


cols_cero = [
    "n_shows_2024",
    "n_cities_2024",
    "n_countries_2024",
    "n_currencies_2024",
    "n_shows_with_capacity_2024",
    "n_shows_2025",
    "n_cities_2025",
    "n_countries_2025",
    "n_currencies_2025",
    "n_shows_with_capacity_2025"
]

for col in cols_cero:
    if col in df_live_agg.columns:
        df_live_agg[col] = df_live_agg[col].fillna(0)

In [12]:
df_live_agg["shows_per_country_2025"] = np.where(
    df_live_agg["n_countries_2025"] > 0,
    df_live_agg["n_shows_2025"] / df_live_agg["n_countries_2025"],
    np.nan
)

df_live_agg["shows_per_country_2024"] = np.where(
    df_live_agg["n_countries_2024"] > 0,
    df_live_agg["n_shows_2024"] / df_live_agg["n_countries_2024"],
    np.nan
)

## columnas bianuales

In [13]:
# columnas combinadas 2024 + 2025
df_live_agg["n_shows_total"] = (
    df_live_agg["n_shows_2024"] + df_live_agg["n_shows_2025"]
)

df_live_agg["n_shows_with_capacity_total"] = (
    df_live_agg["n_shows_with_capacity_2024"] + df_live_agg["n_shows_with_capacity_2025"]
)

df_live_agg["total_capacity_total"] = (
    df_live_agg["total_capacity_2024"].fillna(0) + df_live_agg["total_capacity_2025"].fillna(0)
)

# si no hubo ningún show con capacidad, dejar total_capacity_total como NaN
df_live_agg.loc[
    df_live_agg["n_shows_with_capacity_total"] == 0,
    "total_capacity_total"
] = np.nan

df_live_agg["avg_venue_capacity_total"] = np.where(
    df_live_agg["n_shows_with_capacity_total"] > 0,
    df_live_agg["total_capacity_total"] / df_live_agg["n_shows_with_capacity_total"],
    np.nan
)



In [14]:
def resumir_live_total(df):
    df_total = df[df["event_year"].isin([2024, 2025])].copy()
    
    resumen_total = (
        df_total
        .groupby(["chartmetric_id", "artist_name"], as_index=False)
        .agg(
            n_cities_total=("city_name", lambda x: x.dropna().nunique()),
            n_countries_total=("code2", lambda x: x.dropna().nunique())
        )
    )
    
    return resumen_total

In [15]:
df_live_total = resumir_live_total(df_events)

df_live_agg = df_live_agg.merge(
    df_live_total,
    on=["chartmetric_id", "artist_name"],
    how="left"
)

In [16]:
df_live_agg["shows_per_country_total"] = np.where(
    df_live_agg["n_countries_total"] > 0,
    df_live_agg["n_shows_total"] / df_live_agg["n_countries_total"],
    np.nan
)

In [17]:
print(df_live_agg.shape)
print(df_live_agg.columns.tolist())
df_live_agg.head(10)

(3049, 31)
['chartmetric_id', 'artist_name', 'n_shows_2024', 'avg_low_price_2024', 'avg_high_price_2024', 'avg_venue_capacity_2024', 'total_capacity_2024', 'n_shows_with_capacity_2024', 'n_cities_2024', 'n_countries_2024', 'n_currencies_2024', 'main_currency_2024', 'n_shows_2025', 'avg_low_price_2025', 'avg_high_price_2025', 'avg_venue_capacity_2025', 'total_capacity_2025', 'n_shows_with_capacity_2025', 'n_cities_2025', 'n_countries_2025', 'n_currencies_2025', 'main_currency_2025', 'shows_per_country_2025', 'shows_per_country_2024', 'n_shows_total', 'n_shows_with_capacity_total', 'total_capacity_total', 'avg_venue_capacity_total', 'n_cities_total', 'n_countries_total', 'shows_per_country_total']


In [18]:
print("Duplicados por chartmetric_id:", df_live_agg["chartmetric_id"].duplicated().sum())
print("Duplicados por artist_name:", df_live_agg["artist_name"].duplicated().sum())

Duplicados por chartmetric_id: 0
Duplicados por artist_name: 0


In [19]:
# revierto 0 asignados en las columnas de conteo live
# # que eran necesarios para crear las columnas bianuales

df_live_agg[cols_cero] = df_live_agg[cols_cero].replace(0, np.nan)

In [20]:
df_live_agg.isna().sum().sort_values(ascending=False)

avg_low_price_2024             1641
avg_high_price_2024            1641
n_currencies_2024              1426
main_currency_2024             1426
avg_low_price_2025             1318
avg_high_price_2025            1318
main_currency_2025             1148
n_currencies_2025              1148
total_capacity_2024             832
n_shows_with_capacity_2024      832
avg_venue_capacity_2024         832
n_countries_2024                675
shows_per_country_2024          675
n_cities_2024                   675
n_shows_2024                    653
n_shows_with_capacity_2025      472
total_capacity_2025             472
avg_venue_capacity_2025         472
n_countries_2025                355
shows_per_country_2025          355
n_cities_2025                   355
n_shows_2025                    311
total_capacity_total            154
avg_venue_capacity_total        154
shows_per_country_total          27
artist_name                       0
chartmetric_id                    0
n_shows_with_capacity_total 

# SAVE df_live_agg

In [21]:
ruta_salida_live_agg = r"chartmetric_liveevents_2024_2025_cohorte2\liveevents_2024_2025_artist_agg_cohorte2.csv"
df_live_agg.to_csv(ruta_salida_live_agg, index=False)

In [22]:
df_live_agg.shape


(3049, 31)

In [23]:
df_live_agg.columns


Index(['chartmetric_id', 'artist_name', 'n_shows_2024', 'avg_low_price_2024',
       'avg_high_price_2024', 'avg_venue_capacity_2024', 'total_capacity_2024',
       'n_shows_with_capacity_2024', 'n_cities_2024', 'n_countries_2024',
       'n_currencies_2024', 'main_currency_2024', 'n_shows_2025',
       'avg_low_price_2025', 'avg_high_price_2025', 'avg_venue_capacity_2025',
       'total_capacity_2025', 'n_shows_with_capacity_2025', 'n_cities_2025',
       'n_countries_2025', 'n_currencies_2025', 'main_currency_2025',
       'shows_per_country_2025', 'shows_per_country_2024', 'n_shows_total',
       'n_shows_with_capacity_total', 'total_capacity_total',
       'avg_venue_capacity_total', 'n_cities_total', 'n_countries_total',
       'shows_per_country_total'],
      dtype='str')

In [24]:
df_live_agg.head()


In [25]:
df_live_agg.isna().sum().sort_values(ascending=False).head(15)

avg_low_price_2024            1641
avg_high_price_2024           1641
n_currencies_2024             1426
main_currency_2024            1426
avg_low_price_2025            1318
avg_high_price_2025           1318
main_currency_2025            1148
n_currencies_2025             1148
total_capacity_2024            832
n_shows_with_capacity_2024     832
avg_venue_capacity_2024        832
n_countries_2024               675
shows_per_country_2024         675
n_cities_2024                  675
n_shows_2024                   653
dtype: int64

In [26]:
print("Artistas únicos en eventlevel:", df_events["chartmetric_id"].nunique())
print("Filas en df_live_agg:", df_live_agg.shape[0])

Artistas únicos en eventlevel: 3049
Filas en df_live_agg: 3049


In [27]:
df_live_summary = pd.read_csv(
    r"chartmetric_liveevents_2024_2025_cohorte2\liveevents_2024_2025_summary_cohorte2.csv"
)

In [28]:
print("Artistas en live summary:", df_live_summary["chartmetric_id"].nunique())

Artistas en live summary: 5600


## all artists vs artists con shows

In [29]:
ids_eventlevel = set(df_live_agg["chartmetric_id"].unique())
ids_summary = set(df_live_summary["chartmetric_id"].unique())

faltan_en_eventlevel = ids_summary - ids_eventlevel

print("Artistas en live_summary:", len(ids_summary))
print("Artistas en df_live_agg:", len(ids_eventlevel))
print("Artistas que están en summary pero no en eventlevel:", len(faltan_en_eventlevel))

Artistas en live_summary: 5600
Artistas en df_live_agg: 3049
Artistas que están en summary pero no en eventlevel: 2551


In [30]:
df_faltantes = df_live_summary[
    df_live_summary["chartmetric_id"].isin(faltan_en_eventlevel)
].copy()

print(df_faltantes.shape)
df_faltantes.head(10)

(2551, 13)


In [31]:
print(df_faltantes[["ok", "status_code"]].value_counts(dropna=False))

ok    status_code
True  200            2551
Name: count, dtype: int64


In [32]:
print("Distribución n_shows_2024 en faltantes:")
print(df_faltantes["n_shows_2024"].value_counts(dropna=False).sort_index())

print("\nDistribución n_shows_2025 en faltantes:")
print(df_faltantes["n_shows_2025"].value_counts(dropna=False).sort_index())

Distribución n_shows_2024 en faltantes:
n_shows_2024
0    2551
Name: count, dtype: int64

Distribución n_shows_2025 en faltantes:
n_shows_2025
0    2551
Name: count, dtype: int64


In [33]:
condicion_esperada = (
    (df_faltantes["ok"] == True) &
    (df_faltantes["status_code"] == 200) &
    (df_faltantes["n_shows_2024"] == 0) &
    (df_faltantes["n_shows_2025"] == 0)
)

print("Cantidad de faltantes que cumplen exactamente la condición esperada:", condicion_esperada.sum())
print("Total de faltantes:", len(df_faltantes))

Cantidad de faltantes que cumplen exactamente la condición esperada: 2551
Total de faltantes: 2551


In [34]:
df_casos_raros = df_faltantes[~condicion_esperada].copy()

print(df_casos_raros.shape)
df_casos_raros.head(20)

(0, 13)


,downloaded_at_utc,chartmetric_id,artist_name,ok,status_code,error_text,raw_json_path,n_shows_2024,n_shows_2025,n_requests_live_2024_2025,x_ratelimit_limit,x_ratelimit_remaining,x_ratelimit_reset


## AGREGO BAND al flat.csv

In [35]:
import os
import json
import pandas as pd

JSON_DIR = r"C:\Users\Silvana\Documents\Maestria_Exactas\taller_tesis_1\music_industry_social_live_code\chartmetric_metadata_cohorte2\raw_json_cohorte2"
CSV_PATH = r"C:\Users\Silvana\Documents\Maestria_Exactas\taller_tesis_1\music_industry_social_live_code\chartmetric_metadata_cohorte2\metadata_flat_cohorte2.csv"


# ------------------------------------------
# 1) extraer id + band desde JSON
# ------------------------------------------
rows_band = []

json_files = [f for f in os.listdir(JSON_DIR) if f.lower().endswith(".json")]

for fname in json_files:
    fpath = os.path.join(JSON_DIR, fname)

    try:
        with open(fpath, "r", encoding="utf-8") as f:
            data = json.load(f)

        obj = data.get("obj", data)

        rows_band.append({
            "response_id": obj.get("id"),   # ← clave alineada con tu CSV
            "band": obj.get("band")
        })

    except Exception as e:
        print(f"Error leyendo {fname}: {e}")

df_band = pd.DataFrame(rows_band).drop_duplicates(subset=["response_id"])

# ------------------------------------------
# 2) leer metadata_flat
# ------------------------------------------
df_metadata = pd.read_csv(CSV_PATH)

# ------------------------------------------
# 3) merge limpio (sin tocar nada más)
# ------------------------------------------
df_metadata = df_metadata.drop(columns=["band"], errors="ignore")

df_metadata = df_metadata.merge(
    df_band,
    on="response_id",
    how="left"
)

# ------------------------------------------
# 4) guardar
# ------------------------------------------
df_metadata.to_csv(CSV_PATH, index=False)

print("metadata_flat.csv actualizado con la columna band")
print(df_metadata["band"].value_counts(dropna=False))

metadata_flat.csv actualizado con la columna band
band
False    3969
True     1631
Name: count, dtype: int64


# AGREGO OTROS DATOS DEL JSON
    "num_sp_editorial_playlists": 
    "num_sp_playlists": ,
    "sp_editorial_playlist_total_reach": ,
    "num_am_editorial_playlists": ,
    "num_am_playlists": ,
    "num_de_editorial_playlists": ,
    "num_de_playlists": ,
    "de_playlist_total_reach": ,
    "de_editorial_playlist_total_reach": ,
    "num_az_editorial_playlists": ,
    "num_az_playlists": ,
    "num_yt_editorial_playlists": 
    "num_yt_playlists":
    "yt_playlist_total_reach": 
"tiktok_top_video_views": ,
      "tiktok_top_video_comments": ,
      "tiktok_track_posts": ,

In [36]:
import os
import json
import pandas as pd

JSON_DIR = r"C:\Users\Silvana\Documents\Maestria_Exactas\taller_tesis_1\music_industry_social_live_code\chartmetric_metadata_cohorte2\raw_json_cohorte2"
CSV_PATH = r"C:\Users\Silvana\Documents\Maestria_Exactas\taller_tesis_1\music_industry_social_live_code\chartmetric_metadata_cohorte2\metadata_flat_cohorte2.csv"

# ------------------------------------------
# 1) extraer id + variables desde JSON
# ------------------------------------------
rows_stats = []

json_files = [f for f in os.listdir(JSON_DIR) if f.lower().endswith(".json")]

for fname in json_files:
    fpath = os.path.join(JSON_DIR, fname)

    try:
        with open(fpath, "r", encoding="utf-8") as f:
            data = json.load(f)

        obj = data.get("obj", data)
        cm_stats = obj.get("cm_statistics", {})

        rows_stats.append({
            "response_id": obj.get("id"),

            "num_sp_editorial_playlists": cm_stats.get("num_sp_editorial_playlists"),
            "num_sp_playlists": cm_stats.get("num_sp_playlists"),
            "sp_editorial_playlist_total_reach": cm_stats.get("sp_editorial_playlist_total_reach"),
            "num_am_editorial_playlists": cm_stats.get("num_am_editorial_playlists"),
            "num_am_playlists": cm_stats.get("num_am_playlists"),
            "num_de_editorial_playlists": cm_stats.get("num_de_editorial_playlists"),
            "num_de_playlists": cm_stats.get("num_de_playlists"),
            "de_playlist_total_reach": cm_stats.get("de_playlist_total_reach"),
            "de_editorial_playlist_total_reach": cm_stats.get("de_editorial_playlist_total_reach"),
            "num_az_editorial_playlists": cm_stats.get("num_az_editorial_playlists"),
            "num_az_playlists": cm_stats.get("num_az_playlists"),
            "num_yt_editorial_playlists": cm_stats.get("num_yt_editorial_playlists"),
            "num_yt_playlists": cm_stats.get("num_yt_playlists"),
            "yt_playlist_total_reach": cm_stats.get("yt_playlist_total_reach"),

            "tiktok_top_video_views": cm_stats.get("tiktok_top_video_views"),
            "tiktok_top_video_comments": cm_stats.get("tiktok_top_video_comments"),
            "tiktok_track_posts": cm_stats.get("tiktok_track_posts")
        })

    except Exception as e:
        print(f"Error leyendo {fname}: {e}")

df_stats = pd.DataFrame(rows_stats).drop_duplicates(subset=["response_id"])

# ------------------------------------------
# 2) leer metadata_flat
# ------------------------------------------
df_metadata = pd.read_csv(CSV_PATH)

# ------------------------------------------
# 3) merge limpio
# ------------------------------------------
cols_to_replace = [
    "num_sp_editorial_playlists",
    "num_sp_playlists",
    "sp_editorial_playlist_total_reach",
    "num_am_editorial_playlists",
    "num_am_playlists",
    "num_de_editorial_playlists",
    "num_de_playlists",
    "de_playlist_total_reach",
    "de_editorial_playlist_total_reach",
    "num_az_editorial_playlists",
    "num_az_playlists",
    "num_yt_editorial_playlists",
    "num_yt_playlists",
    "yt_playlist_total_reach",
    "tiktok_top_video_views",
    "tiktok_top_video_comments",
    "tiktok_track_posts"
]

df_metadata = df_metadata.drop(columns=cols_to_replace, errors="ignore")

df_metadata = df_metadata.merge(
    df_stats,
    on="response_id",
    how="left"
)

# ------------------------------------------
# 4) guardar
# ------------------------------------------
df_metadata.to_csv(CSV_PATH, index=False)

print("metadata_flat_cohorte2.csv actualizado con variables de playlists y TikTok")

for col in cols_to_replace:
    print(f"\n{col}")
    print(df_metadata[col].notna().sum(), "valores no nulos")

metadata_flat_cohorte2.csv actualizado con variables de playlists y TikTok

num_sp_editorial_playlists
5591 valores no nulos

num_sp_playlists
5591 valores no nulos

sp_editorial_playlist_total_reach
5591 valores no nulos

num_am_editorial_playlists
5390 valores no nulos

num_am_playlists
5390 valores no nulos

num_de_editorial_playlists
5089 valores no nulos

num_de_playlists
5089 valores no nulos

de_playlist_total_reach
5089 valores no nulos

de_editorial_playlist_total_reach
5089 valores no nulos

num_az_editorial_playlists
4851 valores no nulos

num_az_playlists
4851 valores no nulos

num_yt_editorial_playlists
5196 valores no nulos

num_yt_playlists
5196 valores no nulos

yt_playlist_total_reach
5082 valores no nulos

tiktok_top_video_views
5540 valores no nulos

tiktok_top_video_comments
5540 valores no nulos

tiktok_track_posts
5539 valores no nulos


# MERGE metadata

In [37]:
ruta_metadata = r"chartmetric_metadata_cohorte2\metadata_flat_cohorte2.csv"

df_metadata = pd.read_csv(ruta_metadata)

print(df_metadata.shape)
df_metadata.head()

(5600, 51)


In [38]:
df_metadata = df_metadata.rename(columns={
    "response_id": "chartmetric_id"
})

In [39]:
print("Duplicados en metadata:", df_metadata["chartmetric_id"].duplicated().sum())

Duplicados en metadata: 0


In [40]:
print(df_metadata.columns.tolist())

['downloaded_at_utc', 'requested_chartmetric_id', 'requested_artist_name', 'chartmetric_id', 'name', 'code2', 'pronoun_title', 'gender_title', 'record_label', 'hometown_city', 'cm_artist_rank', 'cm_artist_score', 'career_stage', 'career_stage_score', 'career_trend', 'career_trend_score', 'primary_genre', 'sp_followers', 'sp_monthly_listeners', 'sp_popularity', 'sp_playlist_total_reach', 'ins_followers', 'twitter_followers', 'tiktok_followers', 'tiktok_likes', 'ycs_subscribers', 'ycs_views', 'youtube_daily_video_views', 'youtube_monthly_video_views', 'deezer_fans', 'shazam_count', 'pandora_lifetime_streams', 'pandora_lifetime_stations_added', 'band', 'num_sp_editorial_playlists', 'num_sp_playlists', 'sp_editorial_playlist_total_reach', 'num_am_editorial_playlists', 'num_am_playlists', 'num_de_editorial_playlists', 'num_de_playlists', 'de_playlist_total_reach', 'de_editorial_playlist_total_reach', 'num_az_editorial_playlists', 'num_az_playlists', 'num_yt_editorial_playlists', 'num_yt_pla

In [41]:
df_metadata = df_metadata.rename(columns={
    "name": "artist_name"
})

# MERGE CRITICO

In [42]:
df_final = df_metadata.merge(
    df_live_agg,
    on="chartmetric_id",
    how="left",
    suffixes=("_meta", "_live")
)

In [43]:
[col for col in df_final.columns if "name" in col.lower()]

['requested_artist_name', 'artist_name_meta', 'artist_name_live']

In [44]:
df_final[[
    "chartmetric_id",
    "requested_artist_name",
    "artist_name_meta",
    "artist_name_live"
]].head(15)

In [45]:
df_final["validacion_requested_vs_meta"] = (
    df_final["requested_artist_name"] == df_final["artist_name_meta"]
)

df_final["validacion_meta_vs_live"] = (
    df_final["artist_name_live"].isna() |
    (df_final["artist_name_meta"] == df_final["artist_name_live"])
)

In [46]:
print("Filas metadata:", df_metadata.shape[0])
 #print("Filas metadata sin 34:", df_metadata_sin34.shape[0])

print("Filas final:", df_final.shape[0])

Filas metadata: 5600
Filas final: 5600


## sin info de live events

In [47]:
print("Artistas sin info de live events:", df_final["artist_name_live"].isna().sum())

Artistas sin info de live events: 2551


In [48]:
df_final.loc[df_final["artist_name_live"].isna(), ["chartmetric_id", "artist_name_meta"]].head(10)

In [49]:
print("Duplicados en df_final por chartmetric_id:", df_final["chartmetric_id"].duplicated().sum())

Duplicados en df_final por chartmetric_id: 0


In [50]:
print("Filas metadata:", df_metadata.shape[0])

print("Filas final:", df_final.shape[0])
print("Duplicados en df_final por chartmetric_id:", df_final["chartmetric_id"].duplicated().sum())
print("Artistas sin info de live events:", df_final["artist_name_live"].isna().sum())

Filas metadata: 5600
Filas final: 5600
Duplicados en df_final por chartmetric_id: 0
Artistas sin info de live events: 2551


# DF final

In [51]:
df_final.shape

(5600, 83)

In [52]:
df_final["validacion_meta_vs_live"].value_counts()

validacion_meta_vs_live
True     5599
False       1
Name: count, dtype: int64

In [53]:
df_final.loc[
    df_final["validacion_meta_vs_live"] == False,
    ["chartmetric_id", "artist_name_meta", "artist_name_live"]
]

In [54]:
df_final.columns

Index(['downloaded_at_utc', 'requested_chartmetric_id',
       'requested_artist_name', 'chartmetric_id', 'artist_name_meta', 'code2',
       'pronoun_title', 'gender_title', 'record_label', 'hometown_city',
       'cm_artist_rank', 'cm_artist_score', 'career_stage',
       'career_stage_score', 'career_trend', 'career_trend_score',
       'primary_genre', 'sp_followers', 'sp_monthly_listeners',
       'sp_popularity', 'sp_playlist_total_reach', 'ins_followers',
       'twitter_followers', 'tiktok_followers', 'tiktok_likes',
       'ycs_subscribers', 'ycs_views', 'youtube_daily_video_views',
       'youtube_monthly_video_views', 'deezer_fans', 'shazam_count',
       'pandora_lifetime_streams', 'pandora_lifetime_stations_added', 'band',
       'num_sp_editorial_playlists', 'num_sp_playlists',
       'sp_editorial_playlist_total_reach', 'num_am_editorial_playlists',
       'num_am_playlists', 'num_de_editorial_playlists', 'num_de_playlists',
       'de_playlist_total_reach', 'de_editoria

## elimino columnas innecesarias

In [55]:
cols_drop = [
    "downloaded_at_utc",
    "requested_chartmetric_id",
    "requested_artist_name",
    "artist_name_live",
    "validacion_requested_vs_meta",
    "validacion_meta_vs_live"
]

df_final = df_final.drop(columns=cols_drop)

In [56]:
df_final = df_final.rename(columns={
    "artist_name_meta": "artist_name",
    "code2": "country"
})

In [57]:
print(df_final.shape)
df_final.head()

(5600, 77)


In [58]:
df_final.shape

(5600, 77)

In [59]:
df_final.columns

Index(['chartmetric_id', 'artist_name', 'country', 'pronoun_title',
       'gender_title', 'record_label', 'hometown_city', 'cm_artist_rank',
       'cm_artist_score', 'career_stage', 'career_stage_score', 'career_trend',
       'career_trend_score', 'primary_genre', 'sp_followers',
       'sp_monthly_listeners', 'sp_popularity', 'sp_playlist_total_reach',
       'ins_followers', 'twitter_followers', 'tiktok_followers',
       'tiktok_likes', 'ycs_subscribers', 'ycs_views',
       'youtube_daily_video_views', 'youtube_monthly_video_views',
       'deezer_fans', 'shazam_count', 'pandora_lifetime_streams',
       'pandora_lifetime_stations_added', 'band', 'num_sp_editorial_playlists',
       'num_sp_playlists', 'sp_editorial_playlist_total_reach',
       'num_am_editorial_playlists', 'num_am_playlists',
       'num_de_editorial_playlists', 'num_de_playlists',
       'de_playlist_total_reach', 'de_editorial_playlist_total_reach',
       'num_az_editorial_playlists', 'num_az_playlists',
  

# SAVE  cohorte 2 
(hasta 1500 28 de marzo)
(hasta 3003 31/3/26)
(hasta 4482 3/4/26)

In [ ]:
df_final.to_csv("cohorte2_raw_5600.csv", index=False) #19-4-26


